# ATLAS Group Notebook: Finding the Higgs Boson
## H → γγ: The Diphoton Discovery Channel

**Vanderbilt Programs for Talented Youth | Instructor: Jennifer James, Ph.D. Candidate**

---

### The challenge

The Higgs boson was discovered in 2012 by the ATLAS and CMS experiments at the LHC. One of the two main discovery channels was **H → γγ**: the Higgs decaying to two photons.

This channel is rare. In fact, the Higgs decays to two photons only 0.23% of the time. But it is **clean**: two photons whose invariant mass reconstructs to a sharp peak at 125 GeV, sitting on top of a smooth background.

Your detector is redesigned to find exactly this signal. This notebook walks through the statistical challenge: how do you extract a small signal from a large, smooth background?

The technique you use here is identical to what the real ATLAS analysis did.

---

$$M_{\gamma\gamma} = \sqrt{2 E_1 E_2 (1 - \cos\theta_{12})}$$

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import chi2

np.random.seed(2012)  # The year of the discovery
print("Libraries loaded!")
print("Seed set to 2012, the very year the Higgs was discovered.")

---

## Part 1: Generating the Diphoton Mass Spectrum

We simulate the diphoton invariant mass spectrum as ATLAS would see it:

- **Background**: smooth exponential-like continuum from QCD diphoton production (q+q̄ → γγ, gg → γγ)
- **Signal**: narrow Gaussian peak at 125 GeV from H → γγ. The width is set by detector resolution of ~2 GeV (much wider than the Higgs natural width of 4 MeV)

In [ ]:
# ── Mass range and binning ─────────────────────────────────────────────────────────────
m_bins    = np.linspace(100, 160, 61)   # 1 GeV bins, 100-160 GeV
m_centers = 0.5 * (m_bins[:-1] + m_bins[1:])
m_higgs   = 125.09  # GeV/c2 (PDG 2023 value)

# ── Background model: exponential falling spectrum ───────────────────────────────
def background(m, N_bkg, a, b):
    """Exponential background model."""
    return N_bkg * np.exp(a * (m - 100) + b * (m - 100)**2)

N_bkg  = 8000
a_true = -0.04
b_true = -0.0002

bkg_rate = background(m_centers, N_bkg, a_true, b_true)
bkg_rate = bkg_rate / bkg_rate.sum() * N_bkg

# ── Signal model: Gaussian at 125 GeV ───────────────────────────────────────
N_signal  = 200    # Higgs events (realistic for early LHC run)
sigma_det = 1.8    # GeV (detector mass resolution)

sig_rate = N_signal * np.exp(-0.5 * ((m_centers - m_higgs)/sigma_det)**2)
sig_rate /= sig_rate.sum()
sig_rate *= N_signal

# ── Generate observed data (Poisson fluctuations) ─────────────────────────────
total_rate   = bkg_rate + sig_rate
observed     = np.random.poisson(total_rate)
observed_err = np.sqrt(np.maximum(observed, 1))

print(f"Simulated diphoton mass spectrum:")
print(f"  Total events:    {observed.sum():,}")
print(f"  Background:      ~{int(N_bkg):,}")
print(f"  Signal (Higgs):  ~{N_signal} events at {m_higgs:.2f} GeV")
print(f"  Signal/Total:    {N_signal/(N_bkg+N_signal)*100:.1f}%  (tiny!)")
print(f"  Detector resolution: σ = {sigma_det} GeV")

---

## Part 2: The Raw Spectrum: Can You See the Higgs?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.errorbar(m_centers, observed, yerr=observed_err,
            fmt='o', color='black', markersize=5, capsize=3,
            linewidth=1, label='Data (observed)')
ax.plot(m_centers, bkg_rate + sig_rate, color='firebrick',
        linewidth=2, label='True signal + background (hidden in real life!)', alpha=0.6)
ax.set_xlabel(r'Diphoton invariant mass $M_{\gamma\gamma}$ (GeV/c$^2$)', fontsize=12)
ax.set_ylabel('Events / 1 GeV', fontsize=12)
ax.set_title('Raw diphoton mass spectrum\nCan you see the Higgs by eye?', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(100, 160)
ax.axvline(x=m_higgs, color='gold', linestyle='--', linewidth=1.5, alpha=0.6)

ax = axes[1]
zoom_mask = (m_centers >= 115) & (m_centers <= 135)
ax.errorbar(m_centers[zoom_mask], observed[zoom_mask], yerr=observed_err[zoom_mask],
            fmt='o', color='black', markersize=6, capsize=3, linewidth=1.5, label='Data')
ax.plot(m_centers[zoom_mask], (bkg_rate+sig_rate)[zoom_mask],
        color='firebrick', linewidth=2, label='True total (hidden)', alpha=0.6)
ax.set_xlabel(r'$M_{\gamma\gamma}$ (GeV/c$^2$)', fontsize=12)
ax.set_ylabel('Events / 1 GeV', fontsize=12)
ax.set_title('Zoomed: 115–135 GeV\nThe Higgs signal region', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Simulated ATLAS diphoton spectrum (H → γγ, inspired by 2012 discovery)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("By eye, the Higgs is nearly invisible in the raw spectrum.")
print("The signal is only ~2.4% of the background in the peak region.")
print("We need a background subtraction method to see it.")

---

## Part 3: Background Subtraction: Finding the Bump

The real ATLAS analysis fits a smooth function to the background **excluding** the signal region, then subtracts it. What's left is the signal.

We do the same thing here.

In [ ]:
sideband_mask = (m_centers < 120) | (m_centers > 130)
m_fit   = m_centers[sideband_mask]
obs_fit = observed[sideband_mask]
err_fit = observed_err[sideband_mask]

def bkg_model(m, N, a, b):
    return N * np.exp(a*(m-100) + b*(m-100)**2)

popt, pcov = curve_fit(bkg_model, m_fit, obs_fit, p0=[8000, -0.04, -0.0002],
                        sigma=err_fit, absolute_sigma=True)
perr = np.sqrt(np.diag(pcov))

bkg_fitted = bkg_model(m_centers, *popt)
residuals  = observed - bkg_fitted

fig, axes = plt.subplots(2, 1, figsize=(10, 8),
                          gridspec_kw={'height_ratios': [3, 1]})

ax = axes[0]
ax.errorbar(m_centers, observed, yerr=observed_err,
            fmt='o', color='black', markersize=5, capsize=3, label='Data', zorder=4)
ax.plot(m_centers, bkg_fitted, color='steelblue', linewidth=2.5,
        label='Background fit (sideband method)', zorder=3)
ax.fill_between(m_centers, bkg_fitted - perr[0], bkg_fitted + perr[0],
                alpha=0.2, color='steelblue', label='Fit uncertainty')
ax.axvspan(120, 130, alpha=0.08, color='gold', label='Signal region (excluded from fit)')
ax.axvline(x=m_higgs, color='gold', linestyle='--', linewidth=1.5)
ax.set_ylabel('Events / 1 GeV', fontsize=12)
ax.set_title('Background subtraction: sideband fit method\n'
             'Fit the background outside the signal region, then subtract', fontsize=11)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(100, 160)

ax = axes[1]
ax.errorbar(m_centers, residuals, yerr=observed_err,
            fmt='o', color='firebrick', markersize=5, capsize=3, label='Data - background')
ax.axhline(y=0, color='black', linewidth=1.5)
ax.axvline(x=m_higgs, color='gold', linestyle='--', linewidth=1.5)
ax.axvspan(120, 130, alpha=0.08, color='gold')
ax.set_xlabel(r'$M_{\gamma\gamma}$ (GeV/c$^2$)', fontsize=12)
ax.set_ylabel('Residuals', fontsize=12)
ax.set_title('Residuals = Data - Background fit  →  The Higgs signal!', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(100, 160)

plt.tight_layout()
plt.show()

---

## Part 4: Measuring the Higgs: Signal Extraction

In [ ]:
signal_mask = (m_centers >= 115) & (m_centers <= 135)
m_sig   = m_centers[signal_mask]
res_sig = residuals[signal_mask]
err_sig = observed_err[signal_mask]

def gaussian(m, N, mu, sigma):
    return N * np.exp(-0.5*((m - mu)/sigma)**2)

try:
    popt_sig, pcov_sig = curve_fit(gaussian, m_sig, res_sig,
                                    p0=[150, 125, 2.0], sigma=err_sig,
                                    absolute_sigma=True)
    perr_sig = np.sqrt(np.diag(pcov_sig))
    fit_success = True
except Exception as e:
    print(f"Fit failed: {e}")
    fit_success = False

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(m_sig, res_sig, yerr=err_sig,
            fmt='o', color='black', markersize=7, capsize=4, label='Residuals (data - bkg)')
if fit_success:
    m_fine = np.linspace(115, 135, 200)
    ax.plot(m_fine, gaussian(m_fine, *popt_sig), color='firebrick',
            linewidth=2.5, label='Gaussian fit')
    ax.fill_between(m_fine, 0, gaussian(m_fine, *popt_sig), alpha=0.2, color='firebrick')
ax.axhline(y=0, color='gray', linewidth=1)
ax.axvline(x=m_higgs, color='gold', linestyle='--', linewidth=2,
           label=f'PDG Higgs mass: {m_higgs} GeV')
ax.set_xlabel(r'$M_{\gamma\gamma}$ (GeV/c$^2$)', fontsize=13)
ax.set_ylabel('Events / 1 GeV (background subtracted)', fontsize=12)
ax.set_title('The Higgs boson signal\nBackground-subtracted diphoton mass spectrum', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if fit_success:
    N_fit, mu_fit, sig_fit = popt_sig
    N_err, mu_err, sig_err = perr_sig
    significance = N_fit / N_err
    print(f"Fitted Higgs mass:        {mu_fit:.2f} ± {mu_err:.2f} GeV/c2")
    print(f"Fitted mass resolution:   {abs(sig_fit):.2f} ± {sig_err:.2f} GeV  (detector resolution)")
    print(f"Fitted signal yield:      {N_fit:.0f} ± {N_err:.0f} events")
    print(f"Statistical significance: {significance:.1f}σ")
    print()
    print(f"True Higgs mass (PDG): {m_higgs} GeV/c2")
    print(f"Difference: {abs(mu_fit - m_higgs):.2f} GeV  ({abs(mu_fit-m_higgs)/m_higgs*100:.2f}%)")
    print()
    print("In the 2012 ATLAS discovery paper, the observed significance was 5.9σ.")
    print("5σ is the conventional threshold for a 'discovery' in particle physics.")

---

## Part 5: Why Energy Resolution Matters

Your detector design choices directly affect the width of the Higgs peak, and therefore your ability to see it above the background. Let's see what happens with different detector resolutions.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

resolutions = [
    (1.0, 'Excellent (σ = 1.0 GeV) (crystal calorimeter)'),
    (1.8, 'Good (σ = 1.8 GeV) (ATLAS lead/LAr)'),
    (3.5, 'Poor (σ = 3.5 GeV) (cheap scintillator)'),
    (6.0, 'Very poor (σ = 6.0 GeV) (hadronic calorimeter)'),
]
colors_res = ['steelblue', 'firebrick', 'darkorange', 'gray']

m_fine = np.linspace(115, 135, 300)
for (sigma, label), color in zip(resolutions, colors_res):
    peak = N_signal * np.exp(-0.5*((m_fine - m_higgs)/sigma)**2)
    peak /= peak.max()
    ax.plot(m_fine, peak, linewidth=2.5, color=color,
            linestyle='-' if sigma <= 1.8 else '--', label=label)

ax.axvline(x=m_higgs, color='gold', linestyle=':', linewidth=1.5)
ax.set_xlabel(r'$M_{\gamma\gamma}$ (GeV/c$^2$)', fontsize=13)
ax.set_ylabel('Signal peak (normalized to 1)', fontsize=12)
ax.set_title('Effect of detector energy resolution on the Higgs peak\n'
             'Worse resolution = broader, shorter peak = harder to find', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("With poor resolution, the Higgs peak smears out across many bins.")
print("The signal-to-background ratio in any single bin drops.")
print("At σ = 6 GeV, the Higgs peak would be essentially invisible.")
print()
print("This is why your redesigned detector must prioritize ECAL energy resolution.")

---

## Your Group's Checkpoint Questions

1. In Part 2, the raw spectrum shows ~8,000 background events and only ~200 Higgs signal events in the same mass window. Without the background subtraction, could you claim a discovery? What makes the signal visible despite being only ~2.4% of the total?

2. Look at the sideband fit in Part 3. Why is it important to **exclude** the signal region (120–130 GeV) when fitting the background? What would go wrong if you included it?

3. In Part 5, compare the σ = 1.0 GeV (crystal calorimeter) and σ = 6.0 GeV (hadronic calorimeter) resolution curves. Roughly how much taller is the sharp peak compared to the broad one at the peak center? Why does this matter for signal-to-background?

4. The real ATLAS discovery required 5σ significance, meaning the probability that the signal is a statistical fluctuation is less than 1 in 3.5 million. Your fit gives a significance of a few σ with 200 signal events. What would you need to do to reach 5σ? *(Hint: think about both increasing signal and decreasing background.)*

5. From your subsystem checklist last night, you decided which ATLAS subsystems to keep and which to drop for H → γγ. Does anything in this notebook change your answer? Specifically, does the magnet appear anywhere in the analysis you just did?